# LeetCode #1135 — Connecting Cities With Minimum Cost

**Medium · Python 3 · Graph / Minimum Spanning Tree / Kruskal / Union-Find**  
[Đề gốc LeetCode](https://leetcode.com/problems/connecting-cities-with-minimum-cost/) · [Đối chiếu mô tả và ví dụ](https://leetcode.doocs.org/en/lc/1135/)

```python
class Solution:
    def minimumCost(self, n: int, connections: List[List[int]]) -> int:
        ...
```

Bài Premium: nếu có quyền truy cập đề trên LeetCode, bạn nên đối chiếu lại skeleton và ràng buộc; notebook dùng mô tả tham khảo của LeetCode Wiki.

**Mục tiêu:** từ hiểu thế nào là *nối thông mọi thành phố* → Brute Force → cạnh rẻ nhất hữu ích → **Minimum Spanning Tree** → **Kruskal + Disjoint Set Union (DSU)**.

## 1. Mục tiêu học

1. Phân biệt **đỉnh** (thành phố), **cạnh** (đường nối), **trọng số** (chi phí).
2. Hiểu liên thông và vì sao nối `n` thành phố phải chọn ít nhất `n-1` cạnh.
3. Nhìn được một cạnh tạo **chu trình** mà không tăng khả năng kết nối.
4. Hiểu `sort(key=...)`, `parent`, `find`, `union` từ ví dụ nhỏ.
5. Viết Kruskal với DSU và giải thích vì sao chọn cạnh rẻ nhất không gây chu trình là đúng.
6. So với cách vét cạn trên đồ thị nhỏ; xử lý đồ thị không liên thông.

## 2. Đề bài và ví dụ

Có `n` thành phố gắn số **1 đến n**. Mỗi phần tử `[x,y,cost]` mô tả một kết nối hai chiều giữa thành phố `x` và `y`, có chi phí `cost`.

Chọn một tập các kết nối để **mọi cặp thành phố đều đi tới nhau theo một đường nào đó**, sao cho tổng chi phí nhỏ nhất. Nếu không thể nối thông, trả `-1`.

| Input | Output | Giải thích |
|---|---:|---|
| `n=3, [[1,2,5],[1,3,6],[2,3,1]]` | `6` | Chọn cạnh 2–3 giá 1 và 1–2 giá 5 |
| `n=4, [[1,2,3],[3,4,4]]` | `-1` | Không có đường nối giữa hai nhóm |

**Ràng buộc theo bản tham khảo:** `1<=n<=10^4`; `1<=len(connections)<=10^4`; `1<=x,y<=n`, `x!=y`, `0<=cost<=10^5`.

## 3. Cây và liên thông nhìn bằng tay

`n=3`, các cạnh `[1,2]`, `[2,3]` tạo một **cây**: ba đỉnh, hai cạnh, nối được mọi cặp, không có chu trình. Thêm cạnh `[1,3]` tạo **chu trình 1→2→3→1**.

Với chi phí không âm, nếu một phương án đã liên thông mà chứa chu trình, ta có thể bỏ một cạnh trong chu trình mà vẫn liên thông, không làm tổng chi phí tăng. Vì vậy tồn tại phương án tối ưu là **cây khung** có `n-1` cạnh.

In [1]:
cities={1,2,3}
tree_edges=[(1,2),(2,3)]
cycle_edge=(1,3)
print('Cây ví dụ:',tree_edges,'| cạnh đóng chu trình:',cycle_edge)
assert len(tree_edges)==len(cities)-1

Cây ví dụ: [(1, 2), (2, 3)] | cạnh đóng chu trình: (1, 3)


## 4. Nền tảng Python: List, Tuple, `sorted(..., key=...)`

Mỗi cạnh là list gồm 3 giá trị. `edge[2]` là **chi phí**. `sorted(edges, key=lambda edge: edge[2])` trả một list mới có thứ tự chi phí tăng dần, giữ nguyên list đầu vào.

`lambda edge: edge[2]` là cách viết ngắn của một hàm lấy chi phí để sắp xếp.

In [2]:
edges_demo=[[1,2,5],[1,3,6],[2,3,1]]
by_cost=sorted(edges_demo,key=lambda edge:edge[2])
print('Gốc:',edges_demo)
print('Tăng theo chi phí:',by_cost)
assert by_cost==[[2,3,1],[1,2,5],[1,3,6]]

Gốc: [[1, 2, 5], [1, 3, 6], [2, 3, 1]]
Tăng theo chi phí: [[2, 3, 1], [1, 2, 5], [1, 3, 6]]


## 5. Kiểm tra liên thông bằng DFS/BFS — kiến thức nền

Nếu chỉ biết một tập cạnh đã chọn, hãy dựng danh sách kề `adj`: từ mỗi thành phố, ta thấy các thành phố có đường trực tiếp tới. DFS dùng **stack** hoặc đệ quy để duyệt từ thành phố `1` và đánh dấu `visited`.

Nếu `len(visited)==n` thì tập cạnh kết nối được tất cả. Tập `visited` là một `set` (không chứa phần tử trùng).

In [3]:
def connected(n,edges):
    if n==0:return False
    adj=[[] for _ in range(n+1)]
    for x,y,*rest in edges:
        adj[x].append(y)
        adj[y].append(x)
    seen={1}
    stack=[1]
    while stack:
        u=stack.pop()
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return len(seen)==n

assert connected(3,[[1,2],[2,3]])
assert not connected(4,[[1,2],[3,4]])
print('DFS kiểm tra liên thông: PASS')

DFS kiểm tra liên thông: PASS


## 6. Brute Force: thử các tập cạnh, giữ tổng rẻ nhất

Ý tưởng dễ nghĩ: với `m` cạnh có `2^m` cách *chọn/không chọn* mỗi cạnh. Thử mọi tập con `n-1` cạnh; nếu liên thông, cập nhật chi phí nhỏ nhất.

Để tránh bùng nổ, ta chỉ dùng cách này làm **oracle cho m nhỏ**, không nộp khi `m=10^4`.

In [4]:
from itertools import combinations

def min_cost_bruteforce(n,connections):
    if n==1:return 0
    best=float('inf')
    for chosen in combinations(connections,n-1):
        if connected(n,chosen):
            best=min(best,sum(c for x,y,c in chosen))
    return -1 if best==float('inf') else best

assert min_cost_bruteforce(3,[[1,2,5],[1,3,6],[2,3,1]])==6
assert min_cost_bruteforce(4,[[1,2,3],[3,4,4]])==-1
print('Brute Force oracle: PASS')

Brute Force oracle: PASS


## 7. Bước đột phá: chỉ chọn cạnh rẻ **nếu nó nối hai nhóm khác nhau**

Thử theo chi phí: cạnh `[2,3,1]` nối nhóm `{2}` với `{3}` → chọn. Cạnh `[1,2,5]` nối `{1}` với `{2,3}` → chọn. Bây giờ cả ba thành phố thông nhau; cạnh `[1,3,6]` không cần nữa vì sẽ đóng chu trình.

Thuật toán **Kruskal**: sắp cạnh tăng theo chi phí, rồi **chỉ nhận cạnh khi hai đầu chưa ở chung thành phần liên thông**.

## 8. Union-Find/Disjoint Set Union (DSU) là gì?

Cấu trúc DSU quản lý các **nhóm đỉnh đã được nối**:

- `parent[x]`: một bước đi lên để tìm đại diện của nhóm chứa `x`.
- `find(x)`: tìm **gốc/đại diện** của nhóm.
- `union(a,b)`: nếu hai gốc khác nhau, gộp chúng và trả `True`.

Lúc đầu mọi đỉnh ở nhóm riêng; `parent[i]=i`.

In [5]:
n_demo=4
parent_demo=list(range(n_demo+1))
print('parent ban đầu:',parent_demo)
print('thành phố 1 có gốc:',parent_demo[1])
assert parent_demo==[0,1,2,3,4]

parent ban đầu: [0, 1, 2, 3, 4]
thành phố 1 có gốc: 1


## 9. Tìm đại diện `find` bằng ví dụ nhỏ

Nếu `parent[3]=2` và `parent[2]=1`, cả `3`, `2`, `1` thuộc một nhóm có gốc là `1`.

`find` đi theo `parent` đến khi gặp `parent[root]==root`. Đường đi có thể dài nếu không tối ưu; **path compression (nén đường đi)** gán thẳng các điểm đi qua về gốc.

In [6]:
parent=[0,1,1,2,4]
def find_demo(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]
        x=parent[x]
    return x
root3=find_demo(3)
print('Gốc của 3 =',root3,'; parent sau khi nén =',parent)
assert root3==1 and parent[3]==1

Gốc của 3 = 1 ; parent sau khi nén = [0, 1, 1, 1, 4]


## 10. Gộp hai nhóm `union`

Sau `find(a)` và `find(b)`, nếu gốc bằng nhau nghĩa là **đã nối gián tiếp** → chọn thêm cạnh sẽ tạo chu trình, nên `union` trả `False`.

Nếu hai gốc khác nhau, gộp chúng → `True` và số thành phần giảm 1. Để cây DSU không cao, thường gắn cây thấp vào cây cao (**union by size/rank**).

In [7]:
class DSU:
    def __init__(self,n):
        self.parent=list(range(n+1))
        self.size=[1]*(n+1)
        self.components=n
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]  # nén đường đi
            x=self.parent[x]
        return x
    def union(self,a,b):
        ra,rb=self.find(a),self.find(b)
        if ra==rb:
            return False
        if self.size[ra]<self.size[rb]:
            ra,rb=rb,ra
        self.parent[rb]=ra
        self.size[ra]+=self.size[rb]
        self.components-=1
        return True

dsu_demo=DSU(3)
assert dsu_demo.union(2,3) is True
assert dsu_demo.union(1,2) is True
assert dsu_demo.union(1,3) is False  # đã cùng nhóm
assert dsu_demo.components==1

## 11. Tính Kruskal từ đầu đến cuối

**Bất biến:** trước mỗi cạnh, DSU biểu diễn đúng các nhóm được tạo bởi **những cạnh đã chọn**. Khi hai đầu ở hai nhóm khác nhau, cạnh đó vừa làm giảm số nhóm vừa không tạo chu trình.

Ta sắp cạnh tăng dần, chọn cạnh khi `union=True`, cộng chi phí, đếm số cạnh đã chọn. Khi đạt `n-1` cạnh ⇒ cây đã nối tất cả.

In [8]:
def min_cost_kruskal(n, connections):
    dsu=DSU(n)
    total=0
    chosen=0
    for x,y,cost in sorted(connections,key=lambda e:e[2]):
        if dsu.union(x,y):
            total+=cost
            chosen+=1
            if chosen==n-1:
                return total
    return 0 if n==1 else -1

assert min_cost_kruskal(3,[[1,2,5],[1,3,6],[2,3,1]])==6
assert min_cost_kruskal(4,[[1,2,3],[3,4,4]])==-1

## 12. Trace Kruskal: xem quyết định từng cạnh

Các cột quan trọng: cạnh đang xét, đại diện **trước khi union**, có chọn không, số thành phần còn lại, tổng phí. Khi hai gốc bằng nhau, việc bỏ cạnh là có lý do, không phải bỏ ngẫu nhiên.

In [9]:
def trace_kruskal(n,connections):
    d=DSU(n);total=0
    print('cạnh | gốc 2 đầu trước | chọn? | số nhóm | tổng')
    for x,y,c in sorted(connections,key=lambda z:z[2]):
        roots=(d.find(x),d.find(y))
        choose=d.union(x,y)
        if choose:total+=c
        print(f'({x},{y},{c}) | {roots} | {choose} | {d.components} | {total}')
    return total if d.components==1 else -1

assert trace_kruskal(3,[[1,2,5],[1,3,6],[2,3,1]])==6
print('--- Trường hợp không thể nối hết ---')
assert trace_kruskal(4,[[1,2,3],[3,4,4]])==-1

cạnh | gốc 2 đầu trước | chọn? | số nhóm | tổng
(2,3,1) | (2, 3) | True | 2 | 1
(1,2,5) | (1, 2) | True | 1 | 6
(1,3,6) | (2, 2) | False | 1 | 6
--- Trường hợp không thể nối hết ---
cạnh | gốc 2 đầu trước | chọn? | số nhóm | tổng
(1,2,3) | (1, 2) | True | 3 | 3
(3,4,4) | (3, 4) | True | 2 | 7


## 13. Bảng minh họa Kruskal với ví dụ có chu trình

`n=4` và các cạnh `[(1,2,1),(2,3,2),(1,3,3),(3,4,4)]`:

| Cạnh | Có chọn? | Lý do |
|---|---|---|
| 1–2 giá 1 | Có | Nối hai nhóm |
| 2–3 giá 2 | Có | Nối hai nhóm |
| 1–3 giá 3 | **Không** | 1 và 3 đã liên thông, đóng chu trình |
| 3–4 giá 4 | Có | Nối đỉnh 4 còn tách biệt |

Tổng = `1+2+4 = 7`.

In [10]:
case_cycle=[[1,2,1],[2,3,2],[1,3,3],[3,4,4]]
assert trace_kruskal(4,case_cycle)==7
assert min_cost_bruteforce(4,case_cycle)==7

cạnh | gốc 2 đầu trước | chọn? | số nhóm | tổng
(1,2,1) | (1, 2) | True | 3 | 1
(2,3,2) | (1, 3) | True | 2 | 3
(1,3,3) | (1, 1) | False | 2 | 3
(3,4,4) | (1, 4) | True | 1 | 7


## 14. Vì sao phép chọn Greedy (tham lam) của Kruskal đúng?

**Cut property (tính chất lát cắt):** nếu ta chia tập đỉnh thành hai phía, một cạnh nhẹ nhất nối hai phía là cạnh *an toàn* để đưa vào **một** cây khung nhỏ nhất (cho phép hòa trọng số).

Khi Kruskal chuẩn bị thêm cạnh `(u,v)`, hai đầu đang ở hai thành phần. Xét lát cắt tách một thành phần khỏi phần còn lại. Vì các cạnh đã được duyệt theo thứ tự tăng dần và cạnh nào nối hai nhóm khác nhau sẽ được chọn ngay, cạnh đang xét là một lựa chọn nhẹ nhất thích hợp giữa các nhóm tại bước đó. Ta có thể đổi cạnh tương ứng của một MST nào đó thành cạnh này mà không tăng chi phí.

Lặp lập luận trao đổi này cho mỗi cạnh được chọn ⇒ luôn tồn tại một MST chứa toàn bộ những cạnh đã chọn. Sau `n-1` cạnh, ta được một MST. Nếu duyệt hết mà còn nhiều thành phần, đồ thị gốc không liên thông, không có cách nối.

## 15. Code nộp LeetCode (Kruskal + DSU)

Bản dưới viết logic tự chứa bên trong `Solution`, không phụ thuộc `DSU` của phần giảng. Tránh sửa mảng input ngoài ý muốn bằng `sorted(...)` thay vì `connections.sort(...)`.

In [11]:
from typing import List

class Solution:
    def minimumCost(self, n: int, connections: List[List[int]]) -> int:
        parent=list(range(n+1))
        size=[1]*(n+1)

        def find(x):
            while parent[x]!=x:
                parent[x]=parent[parent[x]]
                x=parent[x]
            return x

        total=0
        selected=0
        for x,y,cost in sorted(connections,key=lambda edge:edge[2]):
            rx,ry=find(x),find(y)
            if rx==ry:
                continue
            if size[rx]<size[ry]:
                rx,ry=ry,rx
            parent[ry]=rx
            size[rx]+=size[ry]
            total+=cost
            selected+=1
            if selected==n-1:
                return total
        return 0 if n==1 else -1

assert Solution().minimumCost(3,[[1,2,5],[1,3,6],[2,3,1]])==6
assert Solution().minimumCost(4,[[1,2,3],[3,4,4]])==-1

## 16. Kiểm thử ca biên

- Đồ thị chỉ có cây (đủ `n-1` cạnh).
- Có các cạnh song song cùng hai đầu, giá khác nhau.
- Có chi phí bằng 0.
- Có thành phố cô lập ⇒ -1.
- `n=1` và không có cạnh: một biến thể tự nhiên để kiểm tra cấu trúc code; **nếu đề quy định phải có ít nhất một cạnh và `x!=y`, trường hợp này ngoài miền input chính thức**.

In [12]:
cases=[
    (3,[[1,2,1],[2,3,2]],3),
    (2,[[1,2,5],[1,2,1]],1),
    (3,[[1,2,0],[2,3,0],[1,3,5]],0),
    (5,[[1,2,1],[2,3,2],[3,4,3]],-1),
    (1,[],0),
]
for n,e,expected in cases:
    actual=Solution().minimumCost(n,e)
    assert actual==expected,(n,e,actual,expected)
    print(n,e,'->',actual,'PASS')

3 [[1, 2, 1], [2, 3, 2]] -> 3 PASS
2 [[1, 2, 5], [1, 2, 1]] -> 1 PASS
3 [[1, 2, 0], [2, 3, 0], [1, 3, 5]] -> 0 PASS
5 [[1, 2, 1], [2, 3, 2], [3, 4, 3]] -> -1 PASS
1 [] -> 0 PASS


## 17. Đối chiếu ngẫu nhiên với Brute Force độc lập

Brute Force thử tổ hợp cạnh kết nối và Kruskal chọn greedy; hai cách tư duy khác nhau. Ta chỉ tạo đồ thị tối đa 5 đỉnh, 8 cạnh để phép vét cạn chạy nhanh.

In [13]:
import random
rng=random.Random(1135)
for case_id in range(250):
    n=rng.randint(2,5)
    all_pairs=[(x,y) for x in range(1,n+1) for y in range(x+1,n+1)]
    m=rng.randint(1,min(8,len(all_pairs)))
    picked=rng.sample(all_pairs,m)
    edges=[[x,y,rng.randint(0,20)] for x,y in picked]
    expected=min_cost_bruteforce(n,edges)
    actual=Solution().minimumCost(n,edges)
    assert actual==expected,(case_id,n,edges,actual,expected)
print('250 đồ thị nhỏ đối chiếu Brute Force: PASS')

250 đồ thị nhỏ đối chiếu Brute Force: PASS


## 18. Kiểm thử đồ thị lớn

Tạo chuỗi `1–2–3–...–10000` (có đúng 9999 cạnh), tổng chi phí `9999`, sau đó thêm một cạnh đắt hơn để thấy thuật toán không chọn dư. Không làm Brute Force ở quy mô này.

In [14]:
n_large=10000
chain=[[i,i+1,1] for i in range(1,n_large)]
chain.append([1,n_large,10**5])
assert Solution().minimumCost(n_large,chain)==n_large-1
print('Đồ thị 10.000 thành phố: PASS')

Đồ thị 10.000 thành phố: PASS


## 19. Một cách khác: thuật toán Prim (mở rộng từ cây đang có)

Kruskal sắp **mọi cạnh** theo trọng số; Prim bắt đầu ở một đỉnh, dùng **min-heap** lấy cạnh rẻ nhất đi từ phần đã thăm sang phần chưa thăm. Mỗi bước thêm đúng một đỉnh mới, không tạo chu trình.

- `heapq.heappush(heap, item)` thêm phần tử.
- `heapq.heappop(heap)` lấy tuple nhỏ nhất theo thứ tự từ trái sang phải (đặt `cost` đầu).
- `visited` ngăn chọn lại thành phố đã nối.

Đây là biến thể để hiểu MST, không bắt buộc cho code nộp.

In [15]:
import heapq

def min_cost_prim(n,connections):
    if n==1:return 0
    adj=[[] for _ in range(n+1)]
    for x,y,c in connections:
        adj[x].append((c,y))
        adj[y].append((c,x))
    heap=[(0,1)]
    seen=set()
    total=0
    while heap and len(seen)<n:
        cost,u=heapq.heappop(heap)
        if u in seen:continue
        seen.add(u)
        total+=cost
        for next_cost,v in adj[u]:
            if v not in seen:
                heapq.heappush(heap,(next_cost,v))
    return total if len(seen)==n else -1

assert min_cost_prim(3,[[1,2,5],[1,3,6],[2,3,1]])==6
assert min_cost_prim(4,[[1,2,3],[3,4,4]])==-1
assert min_cost_prim(4,case_cycle)==7

## 20. Đối chiếu thêm Kruskal, Prim và Brute Force

Một bài có hai thuật toán chuẩn (Kruskal/Prim) và một oracle Brute Force giúp tăng độ tin cậy. Với cạnh có trọng số bằng nhau, danh sách cạnh chọn có thể khác nhưng **chi phí MST tối thiểu** luôn giống.

In [16]:
for test_id in range(160):
    n=rng.randint(2,5)
    pairs=[(a,b) for a in range(1,n+1) for b in range(a+1,n+1)]
    e=[[a,b,rng.randint(0,8)] for a,b in rng.sample(pairs,rng.randint(1,min(8,len(pairs))))]
    expected=min_cost_bruteforce(n,e)
    assert min_cost_prim(n,e)==min_cost_kruskal(n,e)==expected
print('160 test 3 cách giải: PASS')

160 test 3 cách giải: PASS


## 21. Phân tích Time/Space Complexity

Gọi `n` là số thành phố, `m=len(connections)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| Thử các tổ hợp | Có thể tăng theo hàm mũ | tùy cách tạo tổ hợp | chỉ dùng cho test nhỏ |
| Kruskal + DSU | **`O(m log m + m α(n))`**, thường ghi `O(m log m)` | **`O(n+m)`** khi tạo danh sách `sorted` | `α(n)` gần như hằng số |
| Prim + heap | `O((n+m)log(m+1))` | `O(n+m)` | adjacency list + heap |

Với Kruskal, `parent` và `size` tốn `O(n)`, còn `sorted(connections,...)` tạo một list có `m` phần tử. Nếu sắp tại chỗ, bộ nhớ phụ có thể giảm, nhưng phải nêu rõ giả định về thuật toán sắp xếp.

## 22. Sai lầm thường gặp

1. **Chọn luôn `n-1` cạnh rẻ nhất:** chúng có thể tạo chu trình và bỏ đỉnh cô lập.
2. **Cộng tiền kể cả khi `union=False`:** sẽ cộng cạnh không được chọn.
3. **Không kiểm tra đồ thị liên thông:** trả tổng một vài nhóm thay vì `-1`.
4. **Nhầm ID thành phố 1-based với chỉ số 0-based:** tạo `parent` dài `n+1` hoặc trừ `1` nhất quán.
5. **Không nén đường đi / union by size:** vẫn có thể đúng nhưng dễ chậm vì DSU cao.

In [17]:
def wrong_smallest_n_minus_1(n,edges):
    picked=sorted(edges,key=lambda e:e[2])[:n-1]
    return sum(e[2] for e in picked)

wrong=wrong_smallest_n_minus_1(4,case_cycle)
print('Sai nếu chọn 3 cạnh rẻ nhất:',wrong,'; Kruskal đúng:',min_cost_kruskal(4,case_cycle))
assert wrong==6 and min_cost_kruskal(4,case_cycle)==7

Sai nếu chọn 3 cạnh rẻ nhất: 6 ; Kruskal đúng: 7


## 23. Mẹo nhận dạng pattern — khi nào dùng MST?

Những dấu hiệu điển hình:

- Cần **nối thông tất cả các điểm**, chi phí cạnh cho trước.
- Mục tiêu **tổng chi phí nhỏ nhất**, không phải khoảng cách nhỏ nhất từ một nguồn đến từng đích.
- Kết quả có dạng chọn tập cạnh, không yêu cầu thứ tự đi qua.

**Phân biệt:** Bài tìm đường đi ngắn nhất (Dijkstra) tối ưu từng đường từ nguồn; MST tối ưu **tổng trọng số của mạng nối tất cả đỉnh**. Hai bài khác nhau.

## 24. Biến thể — trả cả những cạnh được chọn

Nếu muốn thiết kế đường đi cụ thể, chỉ cần ghi các cạnh mà `union` trả `True`. Hàm này dành cho học/biến thể; đề #1135 chỉ yêu cầu **số tiền**.

In [18]:
def kruskal_with_edges(n,connections):
    d=DSU(n)
    used=[]
    for x,y,c in sorted(connections,key=lambda e:e[2]):
        if d.union(x,y):
            used.append((x,y,c))
    if d.components!=1:return -1,[]
    return sum(c for x,y,c in used),used

cost,selected=kruskal_with_edges(4,case_cycle)
print('Cạnh được chọn:',selected,'| tổng:',cost)
assert cost==7 and len(selected)==3

Cạnh được chọn: [(1, 2, 1), (2, 3, 2), (3, 4, 4)] | tổng: 7


## 25. Bài tập tự luyện (5 câu, dễ → khó)

1. **Tính bằng tay:** từ `n=3` và cạnh `[(1,2,5),(1,3,6),(2,3,1)]`, viết thứ tự Kruskal xét và chọn cạnh.
2. **Dự đoán output:** sau `union(1,2)` rồi `union(2,3)`, `find(1)==find(3)` là gì? `components` còn mấy (n=4)?
3. **TODO:** viết `exercise_connected(n,edges)` dùng danh sách kề và DFS.
4. **TODO:** viết `exercise_mst(n,connections)` dùng DSU và sort theo cost.
5. **Mini challenge:** trả `(chi_phí, danh_sách_cạnh_chọn)` và kiểm tra số cạnh `n-1` khi thành công.

In [19]:
def exercise_connected(n,edges):
    # TODO 1: tạo adj
    # TODO 2: DFS bằng stack, duy trì seen
    # TODO 3: len(seen)==n
    pass

def exercise_mst(n,connections):
    # TODO 1: tạo DSU
    # TODO 2: sort edges by cost
    # TODO 3: union thành công thì cộng cost
    # TODO 4: đủ n-1 cạnh -> total, ngược lại -1
    pass

# Tự làm xong mới bật:
# assert exercise_connected(3,[(1,2),(2,3)])
# assert exercise_mst(3,[[1,2,5],[1,3,6],[2,3,1]])==6

## 26. Gợi ý 3 mức

| Bài | Mức 1 — tư duy | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Cạnh 1 là rẻ nhất | `sorted(..., key=lambda e:e[2])` | Chọn 2–3, sau đó 1–2 |
| 2 | Union tạo nhóm lớn hơn | `find(1)==find(3)` | `True`, còn 2 nhóm |
| 3 | Duyệt được từ 1 tới mọi đỉnh? | `seen`, `stack` | `while stack: ...` |
| 4 | Chọn cạnh rẻ, không tạo chu trình | `find`, `union` | `if dsu.union(x,y): total+=cost` |
| 5 | Ghi nhớ cạnh được chọn | `used=[]` | `used.append((x,y,c))` |

Không cần học thuộc hàm DSU; hãy giải thích được mỗi biến lưu ý nghĩa gì.

## 27. Đáp án tham khảo (chỉ sau bài tập)

**Bài 1:** Xét theo thứ tự 2–3 (1), 1–2 (5), 1–3 (6); chọn hai cạnh đầu, tổng `6`.

**Bài 2:** `find(1)==find(3)` là `True`; khi n=4 có 2 nhóm: `{1,2,3}` và `{4}`.

Bài 3,4 dùng `answer_connected` và `answer_mst` bên dưới; bài 5 xem `kruskal_with_edges` ở phần biến thể.

In [20]:
def answer_connected(n,edges):
    graph=[[] for _ in range(n+1)]
    for a,b,*rest in edges:
        graph[a].append(b)
        graph[b].append(a)
    seen={1}
    stack=[1]
    while stack:
        u=stack.pop()
        for v in graph[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return len(seen)==n

def answer_mst(n,connections):
    d=DSU(n)
    total=0
    for a,b,c in sorted(connections,key=lambda e:e[2]):
        if d.union(a,b):
            total+=c
    return total if d.components==1 else -1

assert answer_connected(3,[(1,2),(2,3)])
assert answer_mst(3,[[1,2,5],[1,3,6],[2,3,1]])==6
assert answer_mst(4,[[1,2,3],[3,4,4]])==-1
print('Đáp án tham khảo: PASS')

Đáp án tham khảo: PASS


## 28. Checklist ngắn trước khi submit

```text
MST → sort cạnh theo chi phí
DSU → find(x), union(x,y)
Khác nhóm? → chọn cạnh + cộng chi phí
Cùng nhóm? → bỏ cạnh vì tạo chu trình
Đủ n-1 cạnh? → trả tổng
Hết cạnh vẫn chưa đủ? → trả -1
```

- [ ] Đúng `minimumCost(n,connections)` và ID thành phố 1-based.
- [ ] Cộng chi phí **chỉ khi union thành công**.
- [ ] Trả `-1` nếu không liên thông.
- [ ] Test cạnh chi phí 0, cạnh song song, chu trình, đỉnh cô lập.
- [ ] Đã đối chiếu `O(m log m)` và nhớ có thêm bộ nhớ cho `sorted`.